# Análisis de Mesa Vibratoria
## Ensayo de Dinámica Estructural

**Asignatura:** Ensayos Dinámicos  
**Objetivo:** Analizar el comportamiento dinámico de estructuras sometidas a excitación sísmica mediante mesa vibratoria

---

## Contenido

1. [Marco Teórico](#1-marco-teórico)
2. [Modelo Matemático – Sistema de 1 GDL](#2-modelo-matemático)
3. [Configuración del Ensayo](#3-configuración-del-ensayo)
4. [Análisis en el Dominio del Tiempo – Método de Newmark](#4-método-de-newmark)
5. [Estimación del Amortiguamiento](#5-estimación-del-amortiguamiento)
6. [Análisis en el Dominio de la Frecuencia](#6-análisis-en-frecuencia)
7. [Función de Respuesta en Frecuencia (FRF)](#7-frf)
8. [Espectros de Respuesta Sísmica](#8-espectros-de-respuesta)
9. [Resultados y Discusión](#9-resultados)
10. [Conclusiones](#10-conclusiones)


In [ ]:
# ============================================================
# Importación de librerías
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.fft import fft, fftfreq
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo de gráficas
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'lines.linewidth': 1.5,
})

print("Librerías cargadas correctamente.")

---
## 1. Marco Teórico

### 1.1 Mesa Vibratoria

Una **mesa vibratoria** (o *shaking table*) es un dispositivo de laboratorio que permite aplicar movimientos controlados en la base de una estructura a escala, simulando la excitación sísmica real. Es una herramienta fundamental en la ingeniería sísmica experimental para:

- Verificar modelos analíticos y numéricos
- Evaluar el comportamiento de estructuras ante sismos
- Identificar frecuencias naturales, modos de vibración y razón de amortiguamiento
- Estudiar fenómenos de resonancia

### 1.2 Sistema de Un Grado de Libertad (1 GDL)

El modelo más simple para representar la respuesta estructural es el **oscilador de 1 GDL**, compuesto por:

- **Masa** $m$ [kg]
- **Rigidez** $k$ [N/m]
- **Amortiguamiento viscoso** $c$ [N·s/m]

Cuando la base del sistema recibe un desplazamiento $u_g(t)$, la ecuación de movimiento en términos del **desplazamiento relativo** $u(t) = u_{abs}(t) - u_g(t)$ es:

$$m\ddot{u}(t) + c\dot{u}(t) + ku(t) = -m\ddot{u}_g(t)$$

Dividiendo entre la masa:

$$\ddot{u}(t) + 2\zeta\omega_n\dot{u}(t) + \omega_n^2 u(t) = -\ddot{u}_g(t)$$

donde:
- $\omega_n = \sqrt{k/m}$ = frecuencia natural circular [rad/s]
- $f_n = \omega_n / (2\pi)$ = frecuencia natural [Hz]
- $\zeta = c / (2m\omega_n)$ = razón de amortiguamiento crítico [-]
- $T_n = 1/f_n$ = período natural [s]


---
## 2. Modelo Matemático

### 2.1 Parámetros del modelo estructural

In [ ]:
# ============================================================
# Parámetros del sistema estructural (1 GDL)
# ============================================================

# --- Propiedades físicas ---
m    = 1.0       # masa [kg]
k    = 394.78    # rigidez [N/m]  → fn ≈ 3.16 Hz
zeta = 0.05      # razón de amortiguamiento (5%)

# --- Propiedades derivadas ---
omega_n = np.sqrt(k / m)                       # frecuencia natural circular [rad/s]
f_n     = omega_n / (2.0 * np.pi)              # frecuencia natural [Hz]
T_n     = 1.0 / f_n                            # período natural [s]
c       = 2.0 * zeta * m * omega_n             # coeficiente de amortiguamiento [N·s/m]
omega_d = omega_n * np.sqrt(1.0 - zeta**2)     # frecuencia amortiguada [rad/s]

print("=" * 45)
print("  PARÁMETROS DEL SISTEMA (1 GDL)")
print("=" * 45)
print(f"  Masa          m  = {m:.3f}  kg")
print(f"  Rigidez       k  = {k:.2f} N/m")
print(f"  Amortiguam.   c  = {c:.4f} N·s/m")
print(f"  ζ             ζ  = {zeta:.2f}  (= {zeta*100:.0f}%)")
print("-" * 45)
print(f"  ωₙ               = {omega_n:.4f} rad/s")
print(f"  fₙ               = {f_n:.4f} Hz")
print(f"  Tₙ               = {T_n:.4f} s")
print(f"  ωd               = {omega_d:.4f} rad/s")
print("=" * 45)

---
## 3. Configuración del Ensayo

### 3.1 Señal de entrada – Acelerograma sintético

Se genera una señal de aceleración de base $\ddot{u}_g(t)$ que simula un registro sísmico. La señal se construye como una superposición de armónicos con envolvente tipo Hann para un total de 20 segundos.

In [ ]:
# ============================================================
# Generación del acelerograma de entrada (señal de la mesa)
# ============================================================
import os
os.makedirs('datos', exist_ok=True)

np.random.seed(42)

dt      = 0.005      # paso de tiempo [s]
t_total = 20.0       # duración total [s]
t       = np.arange(0.0, t_total, dt)
N       = len(t)

# Superposición de armónicos que incluye la frecuencia natural
freqs_in = [0.5, 1.0, f_n, 2.0 * f_n, 4.0, 6.0]  # [Hz]
amps_in  = [0.04, 0.06, 0.15, 0.08, 0.05, 0.03]   # [g]
phases   = np.random.uniform(0.0, 2.0 * np.pi, len(freqs_in))

ag = np.zeros(N)
for f_i, a_i, phi_i in zip(freqs_in, amps_in, phases):
    ag += a_i * np.sin(2.0 * np.pi * f_i * t + phi_i)

# Ruido blanco suave
ag += 0.02 * np.random.randn(N)

# Envolvente de Hann para inicio y fin gradual
env  = np.ones(N)
ramp = int(2.0 / dt)   # 2 s de rampa
env[:ramp]  = np.hanning(2 * ramp)[:ramp]
env[-ramp:] = np.hanning(2 * ramp)[ramp:]
ag *= env

ag_ms2 = ag * 9.81    # [g] → [m/s²]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(t, ag, color='#2c6fad', lw=1.2)
ax.set_xlabel('Tiempo [s]')
ax.set_ylabel('Aceleración [g]')
ax.set_title('Acelerograma de entrada — Mesa vibratoria $\\ddot{u}_g(t)$')
ax.axhline(0.0, color='k', lw=0.8, ls='--')
plt.tight_layout()
plt.savefig('datos/acelerograma_entrada.png', dpi=150)
plt.show()

PGA = np.max(np.abs(ag))
print(f"PGA = {PGA:.4f} g  =  {PGA * 9.81:.4f} m/s²")

---
## 4. Método de Newmark

El **método de Newmark** (1959) es un esquema de integración numérica ampliamente utilizado en dinámica estructural para resolver la ecuación de movimiento paso a paso.

Las ecuaciones de actualización son:

$$\dot{u}_{i+1} = \dot{u}_i + \left[(1-\gamma)\ddot{u}_i + \gamma\ddot{u}_{i+1}\right]\Delta t$$

$$u_{i+1} = u_i + \dot{u}_i\Delta t + \left[(\tfrac{1}{2}-\beta)\ddot{u}_i + \beta\ddot{u}_{i+1}\right]\Delta t^2$$

Con los parámetros $\gamma = 0.5$ y $\beta = 0.25$ se obtiene el **método de la aceleración promedio constante** (incondicionalmente estable).

La rigidez efectiva y la fuerza efectiva son:

$$\hat{k} = k + \frac{\gamma}{\beta\Delta t}c + \frac{1}{\beta\Delta t^2}m$$

$$\hat{p}_{i+1} = -m\ddot{u}_{g,i+1} + m\!\left[\frac{u_i}{\beta\Delta t^2} + \frac{\dot{u}_i}{\beta\Delta t} + \left(\frac{1}{2\beta}-1\right)\ddot{u}_i\right] + c\!\left[\frac{\gamma u_i}{\beta\Delta t} + \left(\frac{\gamma}{\beta}-1\right)\dot{u}_i + \Delta t\!\left(\frac{\gamma}{2\beta}-1\right)\ddot{u}_i\right]$$

In [ ]:
# ============================================================
# Integración numérica — Método de Newmark (β=0.25, γ=0.5)
# ============================================================

def newmark(m, c, k, ag, dt, beta=0.25, gamma=0.5):
    """
    Resuelve la ecuación de movimiento de un sistema de 1 GDL
    sometido a aceleración de base ag(t) mediante el método de Newmark.

    Parámetros
    ----------
    m, c, k : float – masa, amortiguamiento y rigidez
    ag      : ndarray – aceleración de base [m/s²]
    dt      : float – paso de tiempo [s]
    beta, gamma : parámetros de Newmark (por defecto: aceleración promedio)

    Retorna
    -------
    u, v, a : desplazamiento relativo [m], velocidad [m/s],
              aceleración relativa [m/s²]
    """
    N = len(ag)
    u = np.zeros(N)
    v = np.zeros(N)
    a = np.zeros(N)

    # Condición inicial: a(0) de la ecuación de movimiento
    a[0] = (-ag[0] - c * v[0] - k * u[0]) / m

    # Rigidez efectiva (constante a lo largo del tiempo)
    k_eff = (k
             + gamma / (beta * dt) * c
             + 1.0 / (beta * dt**2) * m)

    for i in range(N - 1):
        # Fuerza efectiva en i+1
        p_eff = (
            -m * ag[i + 1]
            + m * (u[i] / (beta * dt**2)
                   + v[i] / (beta * dt)
                   + (1.0 / (2.0 * beta) - 1.0) * a[i])
            + c * (gamma / (beta * dt) * u[i]
                   + (gamma / beta - 1.0) * v[i]
                   + dt * (gamma / (2.0 * beta) - 1.0) * a[i])
        )

        u[i + 1] = p_eff / k_eff

        a[i + 1] = ((u[i + 1] - u[i]) / (beta * dt**2)
                    - v[i] / (beta * dt)
                    - (1.0 / (2.0 * beta) - 1.0) * a[i])

        v[i + 1] = v[i] + dt * ((1.0 - gamma) * a[i] + gamma * a[i + 1])

    return u, v, a


# Ejecutar integración
u, v, a_rel = newmark(m, c, k, ag_ms2, dt)

# Aceleración absoluta
a_abs = a_rel + ag_ms2

print("Integración de Newmark completada.")
print(f"  Desplazamiento máximo relativo : {np.max(np.abs(u)) * 100:.4f} cm")
print(f"  Velocidad máxima relativa       : {np.max(np.abs(v)) * 100:.4f} cm/s")
print(f"  Aceleración absoluta máxima     : {np.max(np.abs(a_abs)) / 9.81:.4f} g")

In [ ]:
# ============================================================
# Gráfica de la respuesta en el tiempo
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(t, u * 100, color='#1a7a4a', label='u(t) — desplaz. relativo')
axes[0].set_ylabel('Desplazamiento [cm]')
axes[0].set_title('Respuesta estructural — Sistema 1 GDL (Método de Newmark)')
axes[0].legend(loc='upper right')

axes[1].plot(t, v * 100, color='#d45f00', label='v(t) — veloc. relativa')
axes[1].set_ylabel('Velocidad [cm/s]')
axes[1].legend(loc='upper right')

axes[2].plot(t, a_abs / 9.81, color='#2c6fad', lw=1.2,
             label='$\\ddot{u}_{abs}$(t)')
axes[2].plot(t, ag, color='gray', lw=0.8, alpha=0.6,
             label='$\\ddot{u}_g$(t)')
axes[2].set_ylabel('Aceleración [g]')
axes[2].set_xlabel('Tiempo [s]')
axes[2].legend(loc='upper right')

for ax in axes:
    ax.axhline(0.0, color='k', lw=0.6, ls='--')

plt.tight_layout()
plt.savefig('datos/respuesta_tiempo.png', dpi=150)
plt.show()

---
## 5. Estimación del Amortiguamiento

### 5.1 Método del Decremento Logarítmico

El **decremento logarítmico** $\delta$ relaciona amplitudes sucesivas en vibración libre:

$$\delta = \frac{1}{n}\ln\!\left(\frac{u_1}{u_{n+1}}\right)$$

La razón de amortiguamiento se obtiene como:

$$\zeta = \frac{\delta}{\sqrt{4\pi^2 + \delta^2}} \approx \frac{\delta}{2\pi} \quad (\zeta \ll 1)$$

In [ ]:
# ============================================================
# Vibración libre — Decremento logarítmico
# ============================================================

def respuesta_libre(m, c, k, u0, v0, dt, t_total):
    """Respuesta analítica de vibración libre de un oscilador de 1 GDL."""
    t_arr   = np.arange(0.0, t_total, dt)
    wn      = np.sqrt(k / m)
    z       = c / (2.0 * m * wn)
    wd      = wn * np.sqrt(1.0 - z**2)
    A       = u0
    B       = (v0 + z * wn * u0) / wd
    u_free  = np.exp(-z * wn * t_arr) * (
        A * np.cos(wd * t_arr) + B * np.sin(wd * t_arr)
    )
    return t_arr, u_free


t_free, u_free = respuesta_libre(m, c, k, u0=0.05, v0=0.0,
                                  dt=dt, t_total=10.0)

# Detectar picos positivos
peaks_idx, _ = signal.find_peaks(u_free, height=1e-4)

zeta_est  = 0.0
delta_log = 0.0
if len(peaks_idx) >= 2:
    u_peaks   = u_free[peaks_idx]
    n_ciclos  = len(peaks_idx) - 1
    delta_log = (1.0 / n_ciclos) * np.log(u_peaks[0] / u_peaks[-1])
    zeta_est  = delta_log / np.sqrt(4.0 * np.pi**2 + delta_log**2)

# Gráfica
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_free, u_free * 100, color='#1a7a4a', label='Vibración libre u(t)')
if len(peaks_idx) >= 2:
    ax.plot(t_free[peaks_idx], u_free[peaks_idx] * 100,
            'ro', ms=6, label='Picos detectados')
    env_pos =  0.05 * np.exp(-zeta * omega_n * t_free) * 100
    ax.plot(t_free,  env_pos, 'r--', lw=1.0,
            label='Envolvente $A\\cdot e^{-\\zeta\\omega_n t}$')
    ax.plot(t_free, -env_pos, 'r--', lw=1.0)

ax.set_xlabel('Tiempo [s]')
ax.set_ylabel('Desplazamiento [cm]')
ax.set_title('Vibración libre — Decremento logarítmico')
ax.legend()
plt.tight_layout()
plt.savefig('datos/vibracion_libre.png', dpi=150)
plt.show()

print(f"Picos detectados           : {len(peaks_idx)}")
print(f"Decremento logarítmico δ   = {delta_log:.5f}")
print(f"ζ estimado (dec. log.)     = {zeta_est:.5f}  ({zeta_est * 100:.2f}%)")
print(f"ζ real del modelo          = {zeta:.5f}  ({zeta * 100:.2f}%)")

### 5.2 Método del Semiancho de Banda (Half-Power Bandwidth)

A partir de la FRF, la razón de amortiguamiento se estima como:

$$\zeta \approx \frac{f_2 - f_1}{2 f_n}$$

donde $f_1$ y $f_2$ son las frecuencias donde la amplitud de la FRF es $1/\sqrt{2}$ del máximo (nivel −3 dB).

---
## 6. Análisis en el Dominio de la Frecuencia

Se aplica la **Transformada Rápida de Fourier (FFT)** tanto a la señal de entrada como a la respuesta del sistema para identificar los contenidos de frecuencia dominantes.

In [ ]:
# ============================================================
# Análisis espectral — FFT
# ============================================================

window    = np.hanning(N)
Ag_fft    = fft(ag_ms2 * window)
U_fft     = fft(u      * window)
freqs_fft = fftfreq(N, d=dt)

mask_pos  = freqs_fft > 0
f_plot    = freqs_fft[mask_pos]
Ag_amp    = 2.0 * np.abs(Ag_fft[mask_pos]) / N
U_amp     = 2.0 * np.abs(U_fft[mask_pos])  / N

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].semilogy(f_plot, Ag_amp, color='gray', lw=1.0,
                 label='|FFT($\\ddot{u}_g$)| [m/s²]')
axes[0].axvline(f_n, color='r', ls='--', lw=1.2,
                label=f'$f_n$ = {f_n:.2f} Hz')
axes[0].set_ylabel('Amplitud [m/s²]')
axes[0].set_title('Espectro de amplitud — Acelerograma de entrada')
axes[0].set_xlim(0, 15)
axes[0].legend()

axes[1].semilogy(f_plot, U_amp * 100, color='#1a7a4a', lw=1.2,
                 label='|FFT(u)| [cm]')
axes[1].axvline(f_n, color='r', ls='--', lw=1.2,
                label=f'$f_n$ = {f_n:.2f} Hz')
axes[1].set_xlabel('Frecuencia [Hz]')
axes[1].set_ylabel('Amplitud [cm]')
axes[1].set_title('Espectro de amplitud — Desplazamiento relativo de la estructura')
axes[1].set_xlim(0, 15)
axes[1].legend()

plt.tight_layout()
plt.savefig('datos/espectros_FFT.png', dpi=150)
plt.show()

# Frecuencia pico
f_dom = f_plot[np.argmax(U_amp)]
print(f"Frecuencia dominante (desplazamiento) : {f_dom:.3f} Hz")
print(f"Frecuencia natural teórica            : {f_n:.3f} Hz")

---
## 7. Función de Respuesta en Frecuencia (FRF)

La **transmisibilidad** del sistema de 1 GDL ante excitación de base se expresa como:

$$T(r) = \frac{|\ddot{U}_{abs}|}{|\ddot{U}_g|} = \sqrt{\frac{1 + (2\zeta r)^2}{(1 - r^2)^2 + (2\zeta r)^2}}$$

donde $r = f / f_n$ es la razón de frecuencias.

In [ ]:
# ============================================================
# FRF — Transmisibilidad analítica
# ============================================================

def transmisibilidad(r, zeta_val):
    """Transmisibilidad aceleración absoluta / aceleración de base."""
    num = 1.0 + (2.0 * zeta_val * r)**2
    den = (1.0 - r**2)**2 + (2.0 * zeta_val * r)**2
    return np.sqrt(num / den)


r_arr = np.linspace(0.01, 4.0, 2000)

fig, ax = plt.subplots(figsize=(10, 5))
zetas_plot = [0.01, 0.02, 0.05, 0.10, 0.20]
colors     = plt.cm.plasma(np.linspace(0.1, 0.85, len(zetas_plot)))

for z_i, col_i in zip(zetas_plot, colors):
    ax.semilogy(r_arr, transmisibilidad(r_arr, z_i),
                color=col_i, lw=1.8, label=f'ζ = {z_i:.2f}')

ax.axvline(1.0, color='k', ls='--', lw=1.0, label='$r = 1$ (resonancia)')
ax.axvline(np.sqrt(2.0), color='gray', ls=':', lw=1.0,
           label=f'$r = \\sqrt{{2}}$ ≈ {np.sqrt(2):.2f}')
ax.axhline(1.0, color='k', ls='-', lw=0.6, alpha=0.4)
ax.set_xlabel('Razón de frecuencias $r = f/f_n$')
ax.set_ylabel('Transmisibilidad $T(r)$')
ax.set_title('FRF — Transmisibilidad de aceleración (Sistema 1 GDL)')
ax.set_xlim(0, 4)
ax.set_ylim(0.05, 50)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('datos/FRF_transmisibilidad.png', dpi=150)
plt.show()

H_zeta = transmisibilidad(r_arr, zeta)
r_max  = r_arr[np.argmax(H_zeta)]
H_max  = np.max(H_zeta)
print(f"Pico de la FRF en r = {r_max:.4f}  →  T_max = {H_max:.3f}")

In [ ]:
# ============================================================
# Estimación de ζ por semiancho de banda (-3 dB)
# ============================================================

H_at_fn   = transmisibilidad(np.array([1.0]), zeta)[0]
nivel_3dB = H_at_fn / np.sqrt(2.0)
H_curve   = transmisibilidad(r_arr, zeta)

crossings = np.where(np.diff(np.sign(H_curve - nivel_3dB)))[0]

zeta_hpb = None
if len(crossings) >= 2:
    f1 = r_arr[crossings[0]]  * f_n
    f2 = r_arr[crossings[-1]] * f_n
    zeta_hpb = (f2 - f1) / (2.0 * f_n)
    print("Método semiancho de banda:")
    print(f"  f₁ = {f1:.4f} Hz,  f₂ = {f2:.4f} Hz")
    print(f"  ζ estimado = {zeta_hpb:.5f}  ({zeta_hpb * 100:.2f}%)")
    print(f"  ζ real     = {zeta:.5f}  ({zeta * 100:.2f}%)")

---
## 8. Espectros de Respuesta Sísmica

El **espectro de respuesta** muestra el valor máximo de la respuesta de un oscilador de 1 GDL con amortiguamiento $\zeta$ constante en función del período natural $T_n$:

$$S_a(T_n, \zeta) = \max_t |\ddot{u}_{abs}(t)|$$
$$S_v(T_n, \zeta) = \max_t |\dot{u}(t)|$$
$$S_d(T_n, \zeta) = \max_t |u(t)|$$

In [ ]:
# ============================================================
# Cálculo de espectros de respuesta sísmica
# ============================================================

def espectro_respuesta(ag_ms2, dt, zeta_val,
                       T_min=0.05, T_max=4.0, n_T=150):
    """
    Calcula Sa, Sv, Sd para un acelerograma y una razón de
    amortiguamiento dada, barriendo períodos naturales.
    """
    T_arr = np.linspace(T_min, T_max, n_T)
    Sa    = np.zeros(n_T)
    Sv    = np.zeros(n_T)
    Sd    = np.zeros(n_T)

    for j, Tj in enumerate(T_arr):
        wn_j  = 2.0 * np.pi / Tj
        k_j   = wn_j**2          # masa unitaria
        c_j   = 2.0 * zeta_val * wn_j
        uj, vj, aj = newmark(1.0, c_j, k_j, ag_ms2, dt)
        a_abs_j     = aj + ag_ms2
        Sd[j] = np.max(np.abs(uj))
        Sv[j] = np.max(np.abs(vj))
        Sa[j] = np.max(np.abs(a_abs_j))

    return T_arr, Sa, Sv, Sd


print("Calculando espectros de respuesta (puede tomar unos segundos)...")
zetas_spec = [0.02, 0.05, 0.10]
espectros  = {}
for z_s in zetas_spec:
    T_arr, Sa, Sv, Sd = espectro_respuesta(ag_ms2, dt, z_s, n_T=150)
    espectros[z_s] = (T_arr, Sa, Sv, Sd)
    print(f"  ζ = {z_s:.2f}  →  Sa_máx = {np.max(Sa) / 9.81:.3f} g  "
          f" Sd_máx = {np.max(Sd) * 100:.3f} cm")
print("Cálculo completado.")

In [ ]:
# ============================================================
# Gráfica de espectros de respuesta
# ============================================================

col_map = {0.02: '#d45f00', 0.05: '#2c6fad', 0.10: '#1a7a4a'}

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)
ax_sa = fig.add_subplot(gs[0, :])
ax_sv = fig.add_subplot(gs[1, 0])
ax_sd = fig.add_subplot(gs[1, 1])

for z_s in zetas_spec:
    T_arr, Sa, Sv, Sd = espectros[z_s]
    col = col_map[z_s]
    ax_sa.plot(T_arr, Sa / 9.81,  color=col, lw=1.8, label=f'ζ = {z_s:.2f}')
    ax_sv.plot(T_arr, Sv * 100,    color=col, lw=1.8, label=f'ζ = {z_s:.2f}')
    ax_sd.plot(T_arr, Sd * 100,    color=col, lw=1.8, label=f'ζ = {z_s:.2f}')

for ax in (ax_sa, ax_sv, ax_sd):
    ax.axvline(T_n, color='k', ls='--', lw=1.2,
               label=f'$T_n$ = {T_n:.3f} s')
    ax.set_xlim(0, 4)

ax_sa.set_ylabel('$S_a$ [g]')
ax_sa.set_title('Espectro de Aceleración $S_a(T, ζ)$')
ax_sa.legend(fontsize=9)

ax_sv.set_xlabel('Período $T_n$ [s]')
ax_sv.set_ylabel('$S_v$ [cm/s]')
ax_sv.set_title('Espectro de Velocidad $S_v(T, ζ)$')
ax_sv.legend(fontsize=9)

ax_sd.set_xlabel('Período $T_n$ [s]')
ax_sd.set_ylabel('$S_d$ [cm]')
ax_sd.set_title('Espectro de Desplazamiento $S_d(T, ζ)$')
ax_sd.legend(fontsize=9)

plt.suptitle('Espectros de Respuesta Sísmica — Acelerograma mesa vibratoria',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('datos/espectros_respuesta.png', dpi=150)
plt.show()

---
## 9. Resultados y Discusión

### 9.1 Resumen de parámetros identificados

In [ ]:
# ============================================================
# Tabla resumen de resultados
# ============================================================

T_arr_5, Sa_5, Sv_5, Sd_5 = espectros[0.05]
Sa_Tn = float(np.interp(T_n, T_arr_5, Sa_5))
Sv_Tn = float(np.interp(T_n, T_arr_5, Sv_5))
Sd_Tn = float(np.interp(T_n, T_arr_5, Sd_5))

print("\n" + "=" * 55)
print("        RESUMEN DE RESULTADOS — MESA VIBRATORIA")
print("=" * 55)
print("\n  PARÁMETROS DEL MODELO")
print(f"  Período natural          Tₙ = {T_n:.4f} s")
print(f"  Frecuencia natural        fₙ = {f_n:.4f} Hz")
print(f"  Razón de amortiguamiento   ζ = {zeta * 100:.1f} %")

print("\n  IDENTIFICACIÓN (métodos experimentales)")
print(f"  Frecuencia dominante FFT      = {f_dom:.4f} Hz")
print(f"  ζ (decremento logarítmico)    = {zeta_est * 100:.2f} %")
if zeta_hpb is not None:
    print(f"  ζ (semiancho de banda)        = {zeta_hpb * 100:.2f} %")

print("\n  RESPUESTA MÁXIMA")
print(f"  PGA                           = {PGA:.4f} g")
print(f"  Desplazamiento máx. relativo  = {np.max(np.abs(u)) * 100:.4f} cm")
print(f"  Aceleración absoluta máxima   = {np.max(np.abs(a_abs)) / 9.81:.4f} g")

print("\n  ESPECTRO EN Tₙ  (ζ = 5%)")
print(f"  Sa(Tₙ) = {Sa_Tn / 9.81:.4f} g")
print(f"  Sv(Tₙ) = {Sv_Tn * 100:.4f} cm/s")
print(f"  Sd(Tₙ) = {Sd_Tn * 100:.4f} cm")
print("=" * 55)

In [ ]:
# ============================================================
# Comparación FRF numérica vs. analítica
# ============================================================

Aa_fft  = fft(a_abs * window)
Aa_amp  = 2.0 * np.abs(Aa_fft[mask_pos]) / N

# Cociente espectral (FRF numérica)
with np.errstate(divide='ignore', invalid='ignore'):
    ratio_num = np.where(Ag_amp > 1e-8, Aa_amp / Ag_amp, np.nan)

# FRF analítica en las mismas frecuencias
r_num   = f_plot / f_n
H_teo   = transmisibilidad(r_num + 1e-12, zeta)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].semilogy(f_plot, ratio_num, color='#2c6fad', lw=1.0, alpha=0.75,
                 label='FRF numérica (cociente FFT)')
axes[0].semilogy(f_plot, H_teo, color='r', lw=2.0, ls='--',
                 label='FRF analítica (teórica)')
axes[0].axvline(f_n, color='k', ls=':', lw=1.0,
                label=f'$f_n$ = {f_n:.2f} Hz')
axes[0].set_xlim(0, 15)
axes[0].set_ylim(0.1, 50)
axes[0].set_xlabel('Frecuencia [Hz]')
axes[0].set_ylabel('|FRF|')
axes[0].set_title('FRF: Numérica vs. Analítica')
axes[0].legend(fontsize=9)

axes[1].plot(T_arr_5, Sa_5 / 9.81, color='#2c6fad', lw=1.8,
             label='$S_a$ (Newmark, ζ = 5%)')
axes[1].axvline(T_n, color='k', ls='--', lw=1.2,
                label=f'$T_n$ = {T_n:.3f} s')
axes[1].set_xlabel('Período $T_n$ [s]')
axes[1].set_ylabel('$S_a$ [g]')
axes[1].set_title('Espectro de Aceleración (ζ = 5%)')
axes[1].set_xlim(0, 4)
axes[1].legend(fontsize=9)

plt.suptitle('Comparación de resultados — Mesa Vibratoria',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('datos/comparacion_resultados.png', dpi=150)
plt.show()

---
## 10. Conclusiones

A partir del análisis de la mesa vibratoria se concluye:

### Identificación dinámica
- La frecuencia natural del sistema fue identificada correctamente mediante la FFT del desplazamiento relativo. La frecuencia dominante coincide con $f_n$ del modelo.
- El **decremento logarítmico** y el **semiancho de banda** producen estimaciones de $\zeta$ consistentes con el valor real del modelo (5%).

### Respuesta dinámica
- El método de integración de **Newmark** ($\beta = 0.25$, $\gamma = 0.5$) es incondicionalmente estable y reproduce fielmente la respuesta dinámica del oscilador de 1 GDL.
- Se observa amplificación de la respuesta estructural respecto a la excitación de base, especialmente en la zona de resonancia ($r = f/f_n \approx 1$).

### Espectros de respuesta
- Los espectros $S_a$, $S_v$ y $S_d$ confirman que a mayor amortiguamiento, menor es la amplificación espectral.
- El valor $S_a(T_n,\,5\%)$ representa la demanda sísmica de aceleración sobre la estructura para el registro utilizado.

### Validación del modelo
- La FRF numérica (cociente espectral FFT) muestra buena concordancia con la FRF analítica teórica, validando tanto el modelo como el procedimiento de integración.
- Para enriquecer el ensayo experimental, se recomienda utilizar excitación de ruido blanco o barridos de frecuencia (*chirp*) con mayor densidad espectral en el rango de interés.


In [ ]:
# Resumen numérico final
print("CONCLUSIONES — VALORES NUMÉRICOS")
print("-" * 45)
print(f"Frecuencia natural identificada    : {f_dom:.3f} Hz")
print(f"Período natural identificado       : {1.0 / f_dom:.3f} s")
print(f"ζ (decremento logarítmico)         : {zeta_est * 100:.2f} %")
if zeta_hpb is not None:
    print(f"ζ (semiancho de banda)             : {zeta_hpb * 100:.2f} %")
print(f"Sa(Tₙ, 5%)                         : {Sa_Tn / 9.81:.4f} g")
print(f"Sd(Tₙ, 5%)                         : {Sd_Tn * 100:.4f} cm")
print(f"Factor de amplificación máx. (FRF) : {H_max:.2f}")

---

## Referencias

1. Chopra, A. K. (2017). *Dynamics of Structures: Theory and Applications to Earthquake Engineering* (5ª ed.). Pearson.
2. Clough, R. W. & Penzien, J. (2003). *Dynamics of Structures* (3ª ed.). Computers & Structures, Inc.
3. Newmark, N. M. (1959). A method of computation for structural dynamics. *Journal of the Engineering Mechanics Division*, 85(EM3), 67–94.
4. Paz, M. & Kim, Y. H. (2019). *Structural Dynamics: Theory and Computation* (6ª ed.). Springer.
5. ASCE/SEI 7-22 (2022). *Minimum Design Loads and Associated Criteria for Buildings and Other Structures*.
